<a href="https://colab.research.google.com/github/Piyush99111/CodeSight-AI/blob/main/notebooks/CodeSight_Gemma4_QLoRA_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U "bitsandbytes>=0.46.1" accelerate peft trl transformers datasets wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 127.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.5/176.5 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 45.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does no

In [2]:
import bitsandbytes as bnb
print(f"bitsandbytes successfully loaded! Version: {bnb.__version__}")

bitsandbytes successfully loaded! Version: 0.50.2


In [2]:
from google.colab import userdata
from huggingface_hub import login, whoami
import wandb

# 1. Authenticate Hugging Face
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
user_info = whoami()
print(f"Hugging Face Connected! Logged in as: {user_info['name']}")

# 2. Authenticate Weights & Biases
wandb_key = userdata.get('WANDB_API_KEY')
wandb.login(key=wandb_key)
print("Weights & Biases Connected!")

Hugging Face Connected! Logged in as: Elysianmind


/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sssyadav06 (sssyadav06-bml-munjal-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Weights & Biases Connected!


In [6]:
from transformers import AutoTokenizer

# We start with Gemma 4 E4B Instruction-Tuned (or swap to "google/gemma-4-31B-it")
MODEL_ID = "google/gemma-4-E4B-it"

# Load the official Gemma 4 tokenizer from Hugging Face
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Ensure padding token is set (required for batched GPU training)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded! Vocabulary size: {len(tokenizer)} tokens")
print(f"Pad Token: '{tokenizer.pad_token}' (ID: {tokenizer.pad_token_id})")
print(f"EOS Token: '{tokenizer.eos_token}' (ID: {tokenizer.eos_token_id})")

Tokenizer loaded! Vocabulary size: 262144 tokens
Pad Token: '<pad>' (ID: 0)
EOS Token: '<eos>' (ID: 1)


In [4]:
from datasets import load_dataset

# Load the dataset from Hugging Face Hub
dataset = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split="train")

# Shuffle and select a 2,000-sample subset for fast, high-impact fine-tuning
# (And split 90% for training, 10% for evaluation/validation)
dataset = dataset.shuffle(seed=42).select(range(2000))
dataset_split = dataset.train_test_split(test_size=0.1, seed=42)

train_data = dataset_split["train"]
eval_data = dataset_split["test"]

print(f"Training samples: {len(train_data)} | Validation samples: {len(eval_data)}")
print("\n--- Sample Raw Data Row ---")
print(train_data[0])

README.md:   0%|          | 0.00/905 [00:00<?, ?B/s]

data/train-00000-of-00001-8b6e212f3e1ece(…): reconstructing file:   0%|          |  0.00B / 11.4MB            

data/train-00000-of-00001-8b6e212f3e1ece(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/18612 [00:00<?, ? examples/s]

Training samples: 1800 | Validation samples: 200

--- Sample Raw Data Row ---
{'instruction': 'Write a Python program to search for a given element in a list.', 'input': '', 'output': 'def search_element(element, list_):\n    for i in range(len(list_)):\n        if element == list_[i]:\n            return i\n    return -1', 'prompt': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nWrite a Python program to search for a given element in a list.\n\n### Input:\n\n\n### Output:\ndef search_element(element, list_):\n    for i in range(len(list_)):\n        if element == list_[i]:\n            return i\n    return -1'}


In [5]:
def format_gemma_chat(example):
    # Combine the instruction and optional input code/context
    user_prompt = f"### Task:\n{example['instruction']}"
    if example.get("input") and example["input"].strip() != "":
        user_prompt += f"\n\n### Context / Input:\n{example['input']}"

    # Structure as a standard multi-turn message list for CodeSight-AI
    messages = [
        {
            "role": "system",
            "content": "You are CodeSight-AI, an expert software engineering assistant. Write clean, modular, well-commented Python code with brief complexity analysis."
        },
        {
            "role": "user",
            "content": user_prompt
        },
        {
            "role": "assistant",
            "content": example["output"]
        }
    ]

    # Convert message list into Gemma's native special token string
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": formatted_text}

# Map the formatting function across both train and eval sets
train_dataset = train_data.map(format_gemma_chat)
eval_dataset = eval_data.map(format_gemma_chat)

print("--- Formatted Prompt Ready for Gemma 4 ---")
print(train_dataset[0]["text"][:600])

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

--- Formatted Prompt Ready for Gemma 4 ---
<bos><|turn>system
You are CodeSight-AI, an expert software engineering assistant. Write clean, modular, well-commented Python code with brief complexity analysis.<turn|>
<|turn>user
### Task:
Write a Python program to search for a given element in a list.<turn|>
<|turn>model
def search_element(element, list_):
    for i in range(len(list_)):
        if element == list_[i]:
            return i
    return -1<turn|>



In [4]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# 1. Define Model ID
MODEL_ID = "google/gemma-4-E4B-it"

# 2. Configure 4-bit NormalFloat (nf4) quantization + A100 bfloat16 math
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                        # Compress base weights to 4-bit
    bnb_4bit_quant_type="nf4",                # NormalFloat4 (optimal for neural weights)
    bnb_4bit_use_double_quant=True,           # Compress quantization constants to save extra VRAM
    bnb_4bit_compute_dtype=torch.bfloat16     # Dequantize to bfloat16 during matrix math
)

# 3. Download and load Gemma 4 onto your A100 GPU
print(f"Loading {MODEL_ID} into A100 VRAM...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",                        # Maps layers directly to the A100 GPU
    torch_dtype=torch.bfloat16,
    attn_implementation="eager"               # Stable attention implementation for Gemma 4
)

# 4. Disable KV-cache during training (only needed during generation)
model.config.use_cache = False

print(f"Model loaded! Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

Loading google/gemma-4-E4B-it into A100 VRAM...


config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model loaded! Memory footprint: 9.23 GB


In [10]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. Prepare model for kbit training
model = prepare_model_for_kbit_training(model)

# 2. Configure LoRA adapter targeting the inner linear layers of Gemma 4
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # Target the inner linear weights of the attention and feedforward layers
    target_modules=[
        "q_proj.linear",
        "k_proj.linear",
        "v_proj.linear",
        "o_proj.linear",
        "gate_proj.linear",
        "up_proj.linear",
        "down_proj.linear"
    ]
)

# 3. Attach LoRA matrices
peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

trainable params: 5,701,632 || all params: 7,946,802,464 || trainable%: 0.0717


In [14]:
import wandb
from datasets import load_dataset
from transformers import AutoTokenizer
from trl import SFTConfig, SFTTrainer

# 1. Reload Tokenizer
MODEL_ID = "google/gemma-4-E4B-it"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Reload Dataset & Remove Conflicting Raw Columns ("prompt", "instruction", etc.)
dataset = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split="train")
dataset = dataset.shuffle(seed=42).select(range(2000))
dataset_split = dataset.train_test_split(test_size=0.1, seed=42)

def format_and_tokenize(example):
    user_prompt = f"### Task:\n{example['instruction']}"
    if example.get("input") and example["input"].strip() != "":
        user_prompt += f"\n\n### Context / Input:\n{example['input']}"
    messages = [
        {"role": "system", "content": "You are CodeSight-AI, an expert software engineering assistant. Write clean, modular, well-commented Python code with brief complexity analysis."},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": example["output"]}
    ]
    formatted_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    # Tokenize directly into input_ids so SFTTrainer doesn't hit pyarrow string formatting bugs
    tokenized = tokenizer(
        formatted_text,
        truncation=True,
        max_length=1024,
        padding=False
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

# Remove the raw string columns so only clean tensors ('input_ids', 'attention_mask', 'labels') remain!
raw_cols = dataset_split["train"].column_names
train_dataset = dataset_split["train"].map(format_and_tokenize, remove_columns=raw_cols)
eval_dataset = dataset_split["test"].map(format_and_tokenize, remove_columns=raw_cols)

print(f"Clean tensor datasets ready! Columns: {train_dataset.column_names}")

# 3. Configure Training Arguments for your A100
training_args = SFTConfig(
    output_dir="./codesight-gemma4-checkpoints",
    per_device_train_batch_size=4,            # Batch size per pass on the A100
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,            # Effective batch size = 4 * 4 = 16
    learning_rate=2e-4,                       # Standard QLoRA learning rate
    lr_scheduler_type="cosine",               # Smooth cosine learning rate decay
    warmup_steps=10,                          # Warm up learning rate over first 10 steps
    max_steps=150,                            # 150 optimization steps (~5-8 mins on A100)
    logging_steps=10,                         # Log Training Loss every 10 steps
    eval_strategy="steps",                    # Run validation on unseen eval_dataset
    eval_steps=50,                            # Log Validation Loss every 50 steps
    save_strategy="steps",
    save_steps=50,
    bf16=True,                                # Native A100 Brain Float 16 precision
    optim="paged_adamw_8bit",                 # Memory-safe 8-bit AdamW optimizer
    report_to="wandb",                        # Streams metrics to your live W&B link
    seed=42
)

# 4. Initialize SFTTrainer & Start Training!
trainer = SFTTrainer(
    model=peft_model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    args=training_args
)

print("Starting QLoRA Fine-Tuning on A100...")
trainer.train()

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Clean tensor datasets ready! Columns: ['input_ids', 'attention_mask', 'labels']


Truncating train dataset:   0%|          | 0/1800 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1800 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Starting QLoRA Fine-Tuning on A100...


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
50,3.127674,3.079797,0.596186,0.627888,170151.000000
100,3.016318,3.079797,0.596186,0.627888,336048.000000
150,3.074125,3.079797,0.596186,0.627888,500880.000000


TrainOutput(global_step=150, training_loss=3.0721544138590495, metrics={'train_runtime': 530.5218, 'train_samples_per_second': 4.524, 'train_steps_per_second': 0.283, 'total_flos': 2.21457534357696e+16, 'train_loss': 3.0721544138590495, 'epoch': 1.3288888888888888})

In [15]:
import torch

# 1. Switch model to evaluation/inference mode and re-enable KV cache
peft_model.eval()
peft_model.config.use_cache = True

# 2. Create an unseen coding interview prompt using our exact CodeSight-AI chat template
test_messages = [
    {
        "role": "system",
        "content": "You are CodeSight-AI, an expert software engineering assistant. Write clean, modular, well-commented Python code with brief complexity analysis."
    },
    {
        "role": "user",
        "content": "### Task:\nWrite an efficient Python function to find the longest palindromic substring in a given string `s`, and include its Time and Space Complexity."
    }
]

# 3. Format with tokenizer (add_generation_prompt=True adds '<|turn>model\n' at the end!)
prompt_text = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

# 4. Generate the response on your A100!
print("Generating response from CodeSight-AI...\n" + "="*50)
with torch.no_grad():
    outputs = peft_model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.2,          # Low temperature for deterministic, bug-free code
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode only the newly generated tokens (slicing off the input prompt)
generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
response = tokenizer.decode(generated_ids, skip_special_tokens=True)
print(response)

Generating response from CodeSight-AI...
As CodeSight-AI, I will provide an efficient and well-documented solution to find the longest palindromic substring.

The most efficient common approach for this problem is the "Expand Around Center" method, which achieves $O(n^2)$ time complexity with $O(1)$ extra space.

```python
def longest_palindromic_substring(s: str) -> str:
    """
    Finds the longest palindromic substring within the given string s.

    This function uses the "Expand Around Center" approach. It iterates through 
    every possible center point (both characters and spaces between characters) 
    and expands outwards to find the longest palindrome centered there.

    Args:
        s: The input string.

    Returns:
        The longest palindromic substring found in s.
    """
    n = len(s)
    if n < 2:
        return s

    start = 0  # Start index of the longest palindrome found so far
    max_len = 1  # Length of the longest palindrome found so far

    def expand

In [16]:
# Define your public Hugging Face repository name under your Elysianmind account
HF_REPO_ID = "Elysianmind/CodeSight-Gemma4-E4B-LoRA"

# 1. Save the LoRA adapter and tokenizer locally in Colab first
peft_model.save_pretrained("./codesight-lora-adapter")
tokenizer.save_pretrained("./codesight-lora-adapter")

# 2. Push both the LoRA adapter weights and the tokenizer to your Hugging Face Hub!
print(f"Pushing trained LoRA adapter to https://huggingface.co/{HF_REPO_ID} ...")
peft_model.push_to_hub(HF_REPO_ID, commit_message="Upload QLoRA fine-tuned Gemma 4 E4B adapter trained on A100")
tokenizer.push_to_hub(HF_REPO_ID, commit_message="Upload Gemma 4 tokenizer")

# 3. Close the Weights & Biases run cleanly
wandb.finish()

print(f"\nSUCCESS! Your model is live at: https://huggingface.co/{HF_REPO_ID}")

Pushing trained LoRA adapter to https://huggingface.co/Elysianmind/CodeSight-Gemma4-E4B-LoRA ...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 43.6kB / 11.4MB            

Validating                    :           |   0%            

...

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp1bhk2vp6/tokenizer.json:  75%|#######4  | 24.0MB / 32.2MB            

Validating                    :           |   0%            

eval/entropy,▁▁▁
eval/loss,▁▁▁
eval/mean_token_accuracy,▁▁▁
eval/num_tokens,▁▅█
eval/runtime,▁█▄
eval/samples_per_second,█▁▅
eval/steps_per_second,█▁▅
train/entropy,▄▄▆▅▆█▄▄▇▄▇▁▂█▅
train/epoch,▁▂▂▃▃▃▄▄▅▅▆▆▆▆▇▇███
train/global_step,▁▁▂▃▃▃▃▄▅▅▅▅▆▇▇▇███
+5,...



SUCCESS! Your model is live at: https://huggingface.co/Elysianmind/CodeSight-Gemma4-E4B-LoRA


In [17]:
!pip install -q sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 565.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 15.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incomp

In [18]:
import chromadb
from sentence_transformers import SentenceTransformer

# 1. Load a fast, high-ranking open-source Embedding Model onto your A100 GPU
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")

# 2. Initialize an in-memory ChromaDB Vector Database collection
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(
    name="codesight_live_docs",
    metadata={"hnsw:space": "cosine"} # Use Cosine Similarity for vector search
)

# 3. Sample "Live / Proprietary" Engineering Documentation that no LLM knows by heart
custom_codebase_docs = [
    {
        "id": "doc_1",
        "title": "HyperCacheClient v3.2 API",
        "content": (
            "Library: `hypercache_sdk` (v3.2). To initialize a distributed cache connection, "
            "use `from hypercache_sdk import HyperNode`. Initialize with "
            "`node = HyperNode(cluster_uri='hc://prod:9090', ttl_seconds=300, compression='lz4')`. "
            "To fetch keys in O(1) batch mode, call `await node.mget_async(key_list, fallback_to_disk=True)`. "
            "Note: `get_batch()` is deprecated as of v3.0 and raises a DeprecatedMethodError."
        )
    },
    {
        "id": "doc_2",
        "title": "QuantumTensor Optimizer Spec",
        "content": (
            "Library: `qtensor_ml`. For low-latency matrix routing on edge devices, import "
            "`from qtensor_ml.routing import SparseRouter`. Instantiate with "
            "`router = SparseRouter(top_k_experts=2, capacity_factor=1.25, precision='fp8_e4m3')`. "
            "Execute routing via `routed_tensors, load_balance_loss = router.dispatch(input_embeddings)`."
        )
    },
    {
        "id": "doc_3",
        "title": "FastStream Telemetry Hook",
        "content": (
            "Library: `faststream_telemetry`. To log p99 latency metrics in microservices, "
            "use the decorator `@trace_Endpoint(bucket_ms=[10, 50, 200], export_Prometheus=True)` "
            "from `faststream_telemetry.hooks`. Never use `time.time()` manually in production routes."
        )
    }
]

# 4. Compute 384-dimensional vector embeddings and store them in ChromaDB
texts = [doc["content"] for doc in custom_codebase_docs]
ids = [doc["id"] for doc in custom_codebase_docs]
metadatas = [{"title": doc["title"]} for doc in custom_codebase_docs]

embeddings = embed_model.encode(texts, normalize_embeddings=True).tolist()

collection.upsert(
    ids=ids,
    documents=texts,
    embeddings=embeddings,
    metadatas=metadatas
)

print(f"Vector Database Ready! Indexed {collection.count()} technical documentation chunks.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector Database Ready! Indexed 3 technical documentation chunks.


In [19]:
def ask_codesight_rag(user_query: str, top_k: int = 1) -> str:
    # 1. Embed the user's question into a 384-D vector
    query_vector = embed_model.encode([user_query], normalize_embeddings=True).tolist()

    # 2. Retrieve the most relevant documentation chunk from ChromaDB
    results = collection.query(
        query_embeddings=query_vector,
        n_results=top_k
    )

    retrieved_doc = results["documents"][0][0]
    retrieved_title = results["metadatas"][0][0]["title"]
    distance = results["distances"][0][0]

    print(f"[RAG Retriever] Matched Doc: '{retrieved_title}' (Cosine Distance: {distance:.4f})")
    print(f"[RAG Context Injected]: {retrieved_doc}\n" + "-"*60)

    # 3. Format using the exact CodeSight-AI chat structure our LoRA adapter was trained on!
    rag_user_prompt = (
        f"### Task:\n{user_query}\n\n"
        f"### Context / Input:\nUse the following official documentation strictly:\n{retrieved_doc}"
    )

    messages = [
        {
            "role": "system",
            "content": "You are CodeSight-AI, an expert software engineering assistant. Write clean, modular, well-commented Python code with brief complexity analysis."
        },
        {
            "role": "user",
            "content": rag_user_prompt
        }
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

    # 4. Generate grounded code with your fine-tuned Gemma 4 LoRA model!
    with torch.no_grad():
        outputs = peft_model.generate(
            **inputs,
            max_new_tokens=768,
            temperature=0.1,      # Very low temperature so it strictly obeys the retrieved docs
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

# Test the complete End-to-End Fine-Tuned LLM + RAG Pipeline!
question = (
    "Write an async Python function using `hypercache_sdk` v3.2 to connect to `hc://prod:9090` "
    "with LZ4 compression and fetch a batch of user IDs in O(1) mode. Explain why we avoid `get_batch()`."
)

rag_answer = ask_codesight_rag(question)
print(rag_answer)

[RAG Retriever] Matched Doc: 'HyperCacheClient v3.2 API' (Cosine Distance: 0.1708)
[RAG Context Injected]: Library: `hypercache_sdk` (v3.2). To initialize a distributed cache connection, use `from hypercache_sdk import HyperNode`. Initialize with `node = HyperNode(cluster_uri='hc://prod:9090', ttl_seconds=300, compression='lz4')`. To fetch keys in O(1) batch mode, call `await node.mget_async(key_list, fallback_to_disk=True)`. Note: `get_batch()` is deprecated as of v3.0 and raises a DeprecatedMethodError.
------------------------------------------------------------
As CodeSight-AI, I will provide the requested asynchronous Python function using the specified library constraints, along with a detailed explanation.

## Solution

```python
import asyncio
from typing import List, Dict, Any
# Assuming hypercache_sdk is installed and available
try:
    from hypercache_sdk import HyperNode, DeprecatedMethodError
except ImportError:
    # Mocking the necessary components for execution context 